# ⚡ FAST: Construction Cost Modeling (Cached Features)

## Engineering: Load cached satellite features (seconds, not hours)
Run `build_sat_features.py` ONCE first, then use this notebook for fast experiments

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

In [ ]:
BASE = Path('d:/Projects/Construction-cost')
DATA_DIR = BASE / 'datasets'

# Load tabular data
train = pd.read_csv(DATA_DIR / 'train_tabular.csv')
eval_df = pd.read_csv(DATA_DIR / 'evaluation_tabular_no_target.csv')

# ⚡ Load cached satellite features (FAST!)
sat_train = pd.read_csv(DATA_DIR / 'sat_features_train.csv')
sat_eval = pd.read_csv(DATA_DIR / 'sat_features_eval.csv')

# Merge
train = train.merge(sat_train, on='data_id')
eval_df = eval_df.merge(sat_eval, on='data_id')

print(f'Train: {train.shape}, Eval: {eval_df.shape}')

In [ ]:
# Preprocessing
train['tropical_cyclone_wind_risk'] = train['tropical_cyclone_wind_risk'].fillna('Low')
eval_df['tropical_cyclone_wind_risk'] = eval_df['tropical_cyclone_wind_risk'].fillna('Low')

bool_cols = ['developed_country', 'landlocked', 'access_to_airport', 'access_to_port', 
             'access_to_highway', 'access_to_railway', 'flood_risk_class']
for col in bool_cols:
    train[col] = (train[col] == 'Yes').astype(int)
    eval_df[col] = (eval_df[col] == 'Yes').astype(int)

risk_map = {'Very Low': 0, 'Low': 1, 'Moderate': 2, 'High': 3, 'Very High': 4}
for col in ['seismic_hazard_zone', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk']:
    train[col] = train[col].map(risk_map)
    eval_df[col] = eval_df[col].map(risk_map)

for col in ['country', 'region_economic_classification', 'koppen_climate_zone']:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    eval_df[col] = le.transform(eval_df[col])

In [ ]:
# Country residual modeling
country_means = train.groupby('country')['construction_cost_per_m2_usd'].mean()
train['country_mean'] = train['country'].map(country_means)
train['target_residual'] = train['construction_cost_per_m2_usd'] - train['country_mean']
eval_df['country_mean'] = eval_df['country'].map(country_means)

print(f"Residual - Mean: {train['target_residual'].mean():.1f}, Std: {train['target_residual'].std():.1f}")

In [ ]:
# Features - DROP GDP
drop_cols = ['data_id', 'geolocation_name', 'quarter_label', 'sentinel2_tiff_file_name', 
             'viirs_tiff_file_name', 'construction_cost_per_m2_usd', 'target_residual',
             'country_mean', 'deflated_gdp_usd']

feature_cols = [c for c in train.columns if c not in drop_cols]
print(f'Features: {len(feature_cols)}')

X = train[feature_cols].fillna(0)
y = train['target_residual']
groups = train['geolocation_name']

In [ ]:
# ⚡ FAST training with GroupKFold
gkf = GroupKFold(n_splits=5)
cv_scores = []
models = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42 + fold,
        verbose=-1
    )
    
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
              callbacks=[lgb.early_stopping(50, verbose=False)])
    
    pred_residual = model.predict(X_val)
    pred_final = pred_residual + train.iloc[val_idx]['country_mean'].values
    true_final = train.iloc[val_idx]['construction_cost_per_m2_usd'].values
    
    mae = mean_absolute_error(true_final, pred_final)
    cv_scores.append(mae)
    models.append(model)
    
    print(f'Fold {fold+1} MAE: {mae:.2f}')

print(f'\n⚡ CV MAE: {np.mean(cv_scores):.2f} ± {np.std(cv_scores):.2f}')

In [ ]:
# Predictions
X_eval = eval_df[feature_cols].fillna(0)
residual_preds = np.mean([m.predict(X_eval) for m in models], axis=0)
final_preds = residual_preds + eval_df['country_mean'].values

print('\n📊 Prediction distribution:')
print(f'Mean: {final_preds.mean():.1f}, Std: {final_preds.std():.1f}')
print(f'Min: {final_preds.min():.1f}, Max: {final_preds.max():.1f}')
print(f'25%: {np.percentile(final_preds, 25):.1f}, 75%: {np.percentile(final_preds, 75):.1f}')

submission = pd.DataFrame({
    'data_id': eval_df['data_id'],
    'construction_cost_per_m2_usd': final_preds
})

submission.to_csv(BASE / 'submission_fast.csv', index=False)
print(f'\n✅ Saved: submission_fast.csv')
submission.head()